In [7]:
from os.path import join, dirname
import os
import sys
import gc
import logging
import argparse
from copy import deepcopy
import numpy as np
import torchshow as ts
import torchvision
from einops import rearrange

import torch
from torch import autocast
from torch.cuda.amp import GradScaler
import torch.nn.functional as F
import torch.distributed as dist
import torch.multiprocessing as mp

sys.path.append(join(dirname(dirname(dirname(os.getcwd())))))
sys.path.append(os.getcwd())

from memoryPerceiver.memory_perceiver.main.configuration import IsaacCfg as cf
from memoryPerceiver.memory_perceiver.core.distributed import setup_distributed, average_gradients, scatter_dummy_src, gather_dummy_src
from memoryPerceiver.memory_perceiver.core.miscellaneous import custom_random, print_cuda_tensors
import memoryPerceiver.memory_perceiver.isaac.trainer as tr
from memoryPerceiver.memory_perceiver.isaac.scaler import RunningStandardScaler
from memoryPerceiver.memory_perceiver.isaac.plot import Plotter
from memoryPerceiver.memory_perceiver.isaac.agent import ActorCriticSimple, ActorCriticPerceiver, create_agent

from memoryPerceiver.memory_perceiver.core.model_net import PerceiverIO, MultiEmbedding, MultiQuery, create_memories
from memoryPerceiver.memory_perceiver.processing.vectorize import patchify, fourier_encode, fourier_encode_vector

from matplotlib import pyplot as plt
from torch import nn

%load_ext autoreload
%autoreload 2

device = 'cuda:0'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
tt = MultiEmbedding(3, {'video':16}, 512, device=device)

In [11]:
out = tt({'scalars': torch.randn(2,1,3, device=device), 'video':torch.randn(2,1,3,5,5,16, device=device)})
print(out.shape)

torch.Size([2, 78, 512])


In [26]:
aa = MultiQuery(3, {'video':16}, 512, device=device)

video


In [28]:
aa(1).shape

torch.Size([4, 512])

In [4]:
m = PerceiverIO(1, device=device).to(device)

In [5]:
mem = create_memories(m, 2, device=device)

In [6]:
out, mem = m({'scalars': torch.ones(2,1,1, device=device)}, mem)
for k, v in out.items():
    print(k, v.shape)

scalars torch.Size([2, 1, 1])


In [ ]:
class RoPENd(torch.nn.Module):
    """N-dimensional Rotary Positional Embedding. https://github.com/limefax/rope-nd?tab=readme-ov-file"""
    def __init__(self, shape, base=10000):
        super(RoPENd, self).__init__()

        channel_dims, feature_dim = shape[:-1], shape[-1]
        k_max = feature_dim // (2 * len(channel_dims))

        assert feature_dim % k_max == 0, f'shape[-1] ({feature_dim}) is not divisible by 2 * len(shape[:-1]) ({2 * len(channel_dims)})'

        # tensor of angles to use
        theta_ks = 1 / (base ** (torch.arange(k_max) / k_max))

        # create a stack of angles multiplied by position
        angles = torch.cat([t.unsqueeze(-1) * theta_ks for t in
                            torch.meshgrid([torch.arange(d) for d in channel_dims], indexing='ij')], dim=-1)

        # convert to complex number to allow easy rotation
        rotations = torch.polar(torch.ones_like(angles), angles)

        # store in a buffer so it can be saved in model parameters
        self.register_buffer('rotations', rotations)

    def forward(self, x):
        # convert input into complex numbers to perform rotation
        x = torch.view_as_complex(x.reshape(*x.shape[:-1], -1, 2))
        pe_x = self.rotations * x
        return torch.view_as_real(pe_x).flatten(-2)

In [4]:
shape = [2,3,4,6]
rop = RoPENd(shape)

In [7]:
rop(torch.ones(2,3,4,6))

tensor([[[[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000, -0.3012,  1.3818],
          [ 1.0000,  1.0000,  1.0000,  1.0000, -1.3254,  0.4932],
          [ 1.0000,  1.0000,  1.0000,  1.0000, -1.1311, -0.8489]],

         [[ 1.0000,  1.0000, -0.3012,  1.3818,  1.0000,  1.0000],
          [ 1.0000,  1.0000, -0.3012,  1.3818, -0.3012,  1.3818],
          [ 1.0000,  1.0000, -0.3012,  1.3818, -1.3254,  0.4932],
          [ 1.0000,  1.0000, -0.3012,  1.3818, -1.1311, -0.8489]],

         [[ 1.0000,  1.0000, -1.3254,  0.4932,  1.0000,  1.0000],
          [ 1.0000,  1.0000, -1.3254,  0.4932, -0.3012,  1.3818],
          [ 1.0000,  1.0000, -1.3254,  0.4932, -1.3254,  0.4932],
          [ 1.0000,  1.0000, -1.3254,  0.4932, -1.1311, -0.8489]]],


        [[[-0.3012,  1.3818,  1.0000,  1.0000,  1.0000,  1.0000],
          [-0.3012,  1.3818,  1.0000,  1.0000, -0.3012,  1.3818],
          [-0.3012,  1.3818,  1.0000,  1.0000, -1.3254,  0.4932],
  

In [44]:
fourier_encode(torch.linspace(-1, 1, 10, device=device), 4, 4).shape

torch.Size([10, 8])

In [33]:
data = torch.ones([2, 3, 4, 10])

In [5]:
a = torch.linspace(1., 5 / 2, 3)
idx = (*((None,) * (3 - 1)), ...)
a[idx]

tensor([[[1.0000, 1.7500, 2.5000]]])

In [ ]:
b, *axis, d, device = *data.shape, data.device
axis_pos = list(map(lambda size: torch.linspace(-1., 1., steps=size, device=device), axis))
pos = torch.stack(torch.meshgrid(*axis_pos, indexing = 'ij'), dim = -1)
enc_pos = fourier_encode_vector(pos, 1000, 1)
enc_pos = rearrange(enc_pos, '... n b -> ... (n b)') # out dim is K*(2b+1), k=nb axis, b=bands
enc_pos = F.pad(enc_pos, (0, d - enc_pos.shape[-1]))
enc_pos.shape

torch.Size([3, 4, 10])

In [35]:
print(enc_pos)

tensor([[[ 8.7423e-08, -1.0000e+00, -1.0000e+00,  8.7423e-08, -1.0000e+00,
          -1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 8.7423e-08, -1.0000e+00, -1.0000e+00, -8.6603e-01,  5.0000e-01,
          -3.3333e-01,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 8.7423e-08, -1.0000e+00, -1.0000e+00,  8.6603e-01,  5.0000e-01,
           3.3333e-01,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 8.7423e-08, -1.0000e+00, -1.0000e+00, -8.7423e-08, -1.0000e+00,
           1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  8.7423e-08, -1.0000e+00,
          -1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  1.0000e+00,  0.0000e+00, -8.6603e-01,  5.0000e-01,
          -3.3333e-01,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  1.0000e+00,  0.0000e+00,  8.6603e-01,  5.0000e-01,
           3.3333

In [22]:
# out: K*(2b+1)
enc_pos.shape

torch.Size([3, 4, 258])

In [27]:
F.pad(enc_pos, (0,2)).shape

torch.Size([3, 4, 260])

In [18]:
axis_pos

[tensor([-1.,  0.,  1.], device='cuda:0'),
 tensor([-1.0000, -0.3333,  0.3333,  1.0000], device='cuda:0')]

In [25]:
pos.shape

torch.Size([3, 4, 2])

In [9]:
pos

tensor([[[-1.0000, -1.0000],
         [-1.0000, -0.3333],
         [-1.0000,  0.3333],
         [-1.0000,  1.0000]],

        [[ 0.0000, -1.0000],
         [ 0.0000, -0.3333],
         [ 0.0000,  0.3333],
         [ 0.0000,  1.0000]],

        [[ 1.0000, -1.0000],
         [ 1.0000, -0.3333],
         [ 1.0000,  0.3333],
         [ 1.0000,  1.0000]]])

In [10]:
enc_pos.shape

torch.Size([3, 4, 2, 3])

In [53]:
tt = [torch.tensor([2.0, 3.0]), torch.tensor([4.0, 5.0, 5.0, 6.0])]

In [55]:
torch.stack(torch.meshgrid(torch.tensor([2.0, 3.0]), torch.tensor([4.0, 5.0, 5.0, 6.0]), indexing = 'ij'), dim = -1)

tensor([[[2., 4.],
         [2., 5.],
         [2., 5.],
         [2., 6.]],

        [[3., 4.],
         [3., 5.],
         [3., 5.],
         [3., 6.]]])

In [13]:
model = Tt()

In [14]:
model.query = torch.ones(4, device=device) + model.latent

In [17]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()

Tt()

In [ ]:
out = model(torch.randn(4, device=device))
loss = F.mse_loss(out, torch.zeros(4, device=device))
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [24]:
print(model.latent)
print(model.query)

Parameter containing:
tensor([0.9972, 0.9974, 0.9972, 0.9976], device='cuda:0', requires_grad=True)
tensor([2., 2., 2., 2.], device='cuda:0', grad_fn=<AddBackward0>)
